[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C05_Safety_Evals_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与统计冒烟测试

配套 [00_overview.html](00_overview.html) · <span style="color:#888">MODULE 00 / 8 · 纯 CPU</span>

本 notebook 做三件事：

1. **版本自检**：确认 conda env `safety` 中课程依赖齐全（缺失项标红）；
2. **设备与 API key 检测**：确认 device 与可选 key 的状态（缺 key 完全不影响本课）;
3. **统计冒烟测试**：用一次完全合成的"安全评测"模拟，热身本课最核心的两个工具——
   **two-sided 错误率**（过拒 / 欠拒）与 **Wilson 置信区间**（✏️ 练习 1 / 2）。

> 🛑 **立场声明**：全课防御 / 测量 / 治理视角。本 notebook 中的"请求"只是带标签的占位字符串
> （如 `UNSAFE_REQ_007`），不含任何真实内容；"分类器"是合成分布的 mock。我们练的是测量方法论。

In [ ]:
# ── 1. 版本自检：缺失项标红 ─────────────────────────────────
import importlib, os, sys

RED, GREEN, YELLOW, RESET = "\033[91m", "\033[92m", "\033[93m", "\033[0m"
print(f"Python {sys.version.split()[0]}  ({sys.executable})\n")

pkgs = [  # (import 名, 显示名)
    ("numpy",        "numpy"),
    ("scipy",        "scipy"),
    ("statsmodels",  "statsmodels"),
    ("sklearn",      "scikit-learn"),
    ("pandas",       "pandas"),
    ("matplotlib",   "matplotlib"),
    ("torch",        "torch（CPU 版即可）"),
    ("transformers", "transformers"),
]
missing = []
for mod, name in pkgs:
    try:
        m = importlib.import_module(mod)
        print(f"{GREEN}✔{RESET} {name:<22} {getattr(m, '__version__', '?')}")
    except ImportError:
        missing.append(mod)
        print(f"{RED}✘ {name:<22} 缺失{RESET}")

# ── 2. device + API key（均为可选项，False 不影响本课）─────────
try:
    import torch
    device = "cuda" if torch.cuda.is_available() else (
             "mps" if torch.backends.mps.is_available() else "cpu")
except ImportError:
    device = "cpu (torch 未安装)"

HAS_OPENAI_KEY    = bool(os.environ.get("OPENAI_API_KEY"))
HAS_ANTHROPIC_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))

print(f"\ndevice = {device}   （本课几乎全程 cpu 即可）")
print(f"HAS_OPENAI_KEY    = {HAS_OPENAI_KEY}")
print(f"HAS_ANTHROPIC_KEY = {HAS_ANTHROPIC_KEY}   （可选；缺 key 的 cell 会自动跳过）")

if missing:
    print(f"\n{RED}缺 {len(missing)} 个包，请在 safety 环境中执行：pip install -r ../requirements.txt{RESET}")
else:
    print(f"\n{GREEN}✅ 环境就绪{RESET}")

## 统计冒烟测试：一次"安全评测"的最小模拟

安全评测的错误天然是**双侧的（two-sided）**——这是它与普通分类评测最重要的区别之一：

- **过拒（over-refusal）**：安全请求被错误拒绝 → 伤害模型的有用性（usefulness）；
- **欠拒（under-refusal）**：不安全请求被错误放行 → 伤害安全性（safety）。

调高拒绝阈值，欠拒降、过拒升；调低则反之。**只报告其中一侧的安全评测都是误导**
（"拒绝了 100% 的不安全请求"可以靠拒绝一切来实现）。

下面用合成数据演示：100 条占位请求（80 安全 / 20 不安全），一个用 Beta 分布
打"风险分"的 mock 分类器，看不同阈值下两个错误率怎么此消彼长。

In [ ]:
# ── 统计冒烟：合成请求 + mock 分类器 + two-sided 错误率 ──────
import numpy as np

rng = np.random.default_rng(42)
N_SAFE, N_HARMFUL = 80, 20

# 合成"请求"：只有 id 与标签，内容全部是良性占位符
requests  = [f"SAFE_REQ_{i:03d}" for i in range(N_SAFE)] + \
            [f"UNSAFE_REQ_{i:03d}" for i in range(N_HARMFUL)]
y_harmful = np.array([0] * N_SAFE + [1] * N_HARMFUL)   # 1 = 不安全（标签为模拟设定）

# mock 分类器：合成"风险分"——安全请求偏低、不安全请求偏高，但分布有重叠（现实如此）
risk_score = np.where(y_harmful == 1,
                      rng.beta(5, 2, size=len(y_harmful)),   # 不安全：均值 ~0.71
                      rng.beta(2, 5, size=len(y_harmful)))   # 安全：  均值 ~0.29

print("示例：", requests[0], requests[-1], "\n")
print(f"{'阈值':>4} | {'过拒率 over-refusal':<20} | {'欠拒率 under-refusal':<20}")
print("-" * 56)
for thr in [0.2, 0.4, 0.6, 0.8]:
    y_refused = (risk_score >= thr).astype(int)        # 风险分过线 → 拒绝
    over  = ((y_refused == 1) & (y_harmful == 0)).sum() / N_SAFE
    under = ((y_refused == 0) & (y_harmful == 1)).sum() / N_HARMFUL
    print(f"{thr:>4.1f} | {over:<20.3f} | {under:<20.3f}")

# 观察：阈值升高 → 过拒降、欠拒升。两个错误率必须同时报告。

## ✏️ 练习 1：实现 `two_sided_error_rates`

把上面的双侧错误率封装成函数——它会在模块 04（拒绝校准）、06（监控评估）反复出现。

**任务**：实现 `two_sided_error_rates(y_true_harmful, y_refused)`，返回元组
`(over_refusal_rate, under_refusal_rate)`：

- `over_refusal_rate` = **安全请求**（`y_true_harmful == 0`）中**被拒绝**（`y_refused == 1`）的比例；
- `under_refusal_rate` = **不安全请求**（`y_true_harmful == 1`）中**被放行**（`y_refused == 0`）的比例。

**提示**：注意两个比率的**分母不同**（分别是安全请求数、不安全请求数）——这正是
precision/recall 式"分母意识"的练习。`np.asarray` + 布尔索引约 5 行可完成。

In [ ]:
import numpy as np

def two_sided_error_rates(y_true_harmful, y_refused):
    # 输入：两个等长 0/1 序列（list 或 np.ndarray）
    # 返回：(over_refusal_rate, under_refusal_rate)
    y_true_harmful = np.asarray(y_true_harmful)
    y_refused      = np.asarray(y_refused)
    # TODO: over_refusal_rate  = 安全请求(==0) 中被拒绝(==1) 的比例
    # TODO: under_refusal_rate = 不安全请求(==1) 中被放行(==0) 的比例
    raise NotImplementedError("完成上面两个 TODO 后删除此行")

In [ ]:
# ── 练习 1 自测 ──────────────────────────────────────────────
over, under = two_sided_error_rates([0, 0, 0, 0, 1, 1], [1, 0, 0, 0, 1, 0])
assert abs(over - 0.25) < 1e-9, f"over 应为 1/4，得到 {over}"
assert abs(under - 0.50) < 1e-9, f"under 应为 1/2，得到 {under}"

over, under = two_sided_error_rates([0, 0, 1, 1], [0, 0, 1, 1])   # 完美分类器
assert over == 0.0 and under == 0.0

over, under = two_sided_error_rates([0, 0, 1, 1], [1, 1, 1, 1])   # 拒绝一切
assert over == 1.0 and under == 0.0, "拒绝一切：欠拒=0 但过拒=1 —— 单侧指标的陷阱"

over, under = two_sided_error_rates([0, 0, 1, 1], [0, 0, 0, 0])   # 放行一切
assert over == 0.0 and under == 1.0

print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `wilson_ci`（全课贯穿使用）

评测出的每个比率都来自有限样本，**必须带置信区间**。本课统一使用
**Wilson score interval**（推导见 00_overview.html 第 5 节）：

$$ \mathrm{CI} = \frac{\hat p + \frac{z^2}{2n} \pm z\sqrt{\frac{\hat p (1-\hat p)}{n} + \frac{z^2}{4 n^2}}}{1 + z^2/n}, \qquad \hat p = k/n $$

**任务**：实现 `wilson_ci(k, n, z=1.96)`，返回 `(lo, hi)`。

**提示**：先算 `p_hat`、`denom = 1 + z**2/n`，再算 `center` 与 `margin`，
返回 `(center - margin, center + margin)`。`math.sqrt` 即可，约 6 行。

**为什么不用正态近似** $\hat p \pm z\sqrt{\hat p(1-\hat p)/n}$：当 $k=0$（红队零发现）时它给出
宽度为零的区间 $[0,0]$ ——"绝对安全"的假象；Wilson 仍给出合理的非零上界。

In [ ]:
import math

def wilson_ci(k, n, z=1.96):
    # 输入：n 次试验中 k 次"事件"；返回 (lo, hi)
    # TODO: p_hat = k / n
    # TODO: denom = 1 + z**2 / n
    # TODO: center = (p_hat + z**2 / (2*n)) / denom
    # TODO: margin = z * sqrt( p_hat*(1-p_hat)/n + z**2/(4*n**2) ) / denom
    raise NotImplementedError("完成上面的 TODO 后删除此行")

In [ ]:
# ── 练习 2 自测 ──────────────────────────────────────────────
lo, hi = wilson_ci(3, 10)                       # 10 次中 3 次事件
assert abs(lo - 0.1078) < 1e-3, f"lo 应约 0.108，得到 {lo}"
assert abs(hi - 0.6032) < 1e-3, f"hi 应约 0.603，得到 {hi}"

lo0, hi0 = wilson_ci(0, 10)                     # 零事件：下界为 0，上界仍明显非零！
assert abs(lo0) < 1e-9 and abs(hi0 - 0.2775) < 1e-3, (lo0, hi0)

lo1, hi1 = wilson_ci(10, 10)                    # 全事件：与零事件镜像对称
assert abs(hi1 - 1.0) < 1e-9 and abs(lo1 - (1 - hi0)) < 1e-9, (lo1, hi1)

lo5, hi5 = wilson_ci(5, 10)
assert lo5 < 0.5 < hi5                          # 区间必须覆盖点估计

print("✅ 练习 2 通过")
print(f"\n红队 10 次尝试 0 次成功 → 95% CI 上界仍有 {hi0:.1%} —— 这就是为什么"
      f"\n『我们没发现问题』≠『没有问题』，必须报告区间而非点估计。")

## 📖 参考答案

先自己做，自测全绿后再对照。每题独立 cell，可直接运行覆盖你的实现。

In [ ]:
# 参考答案 · 练习 1（先自己做，再对照）
import numpy as np

def two_sided_error_rates(y_true_harmful, y_refused):
    y_true_harmful = np.asarray(y_true_harmful).astype(bool)
    y_refused      = np.asarray(y_refused).astype(bool)
    safe    = ~y_true_harmful
    harmful = y_true_harmful
    over_refusal_rate  = (y_refused & safe).sum() / safe.sum()        # 分母：安全请求数
    under_refusal_rate = (~y_refused & harmful).sum() / harmful.sum() # 分母：不安全请求数
    return float(over_refusal_rate), float(under_refusal_rate)

print(two_sided_error_rates([0, 0, 0, 0, 1, 1], [1, 0, 0, 0, 1, 0]))  # (0.25, 0.5)

In [ ]:
# 参考答案 · 练习 2（先自己做，再对照）
import math

def wilson_ci(k, n, z=1.96):
    p_hat  = k / n
    denom  = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    margin = z * math.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2)) / denom
    return center - margin, center + margin

print([round(v, 4) for v in wilson_ci(3, 10)])   # [0.1078, 0.6032]
print([round(v, 4) for v in wilson_ci(0, 10)])   # [0.0, 0.2775]

## 各模块算力一览

| 模块 | 算力 | 说明 |
|---|---|---|
| 00 总览与环境 | <strong>CPU</strong> | 本 notebook，秒级 |
| 01 风险分类与安全框架 | <strong>CPU</strong> | 阈值/tripwire 模拟，numpy |
| 02 危险能力评估设计 | <strong>CPU</strong> | 难度阶梯与 uplift 模拟，统计为主 |
| 03 红队方法论 | <strong>CPU</strong> | 发现曲线、覆盖率估计，蒙特卡洛 |
| 04 鲁棒性与拒绝校准 | <strong>CPU</strong>（GPU/MPS 可选） | 可选 Qwen2.5-0.5B 实测 cell，带确定性 mock 回退 |
| 05 Sandbagging 与评测完整性 | <strong>CPU</strong> | 能力隐藏的统计检测，模拟数据 |
| 06 AI Control 与监控 | <strong>CPU</strong> | audit budget / 监控策略的蒙特卡洛 |
| 07 Safety Case 与治理报告 | <strong>CPU</strong> | 证据聚合与报告生成 |

---

环境就绪、两道练习全绿后，进入
**[模块 01 · 风险分类与安全框架](../01_risk_frameworks/01_讲解.html)** ——
先搞清楚"测什么、为什么测、过线了怎么办"，再谈怎么测。